## Score - 0.815

In [125]:
import numpy as np 
import pandas as pd  
import os
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, classification_report

from xgboost import XGBClassifier

In [126]:
pd.read_csv('/kaggle/input/datasets/vedikagupta0/climate-risk-health-prediction-challenge/SampleSubmission.csv').sample(5)

,ID,TargetF1,TargetRAUC
674,ID_BFF5DE26,0,0
717,ID_38EC8944,0,0
958,ID_7F3812BB,0,0
847,ID_7395017A,0,0
630,ID_95F38480,0,0


In [127]:
data_dict =pd.read_csv('/kaggle/input/datasets/vedikagupta0/climate-risk-health-prediction-challenge/data_dictionary.csv')

In [128]:
df = pd.read_csv('/kaggle/input/datasets/vedikagupta0/climate-risk-health-prediction-challenge/Train.csv')

In [129]:
df["is_climate_sensitive"].value_counts()

is_climate_sensitive
1    2047
0    1099
Name: count, dtype: int64

In [130]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3146 entries, 0 to 3145
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   ID                    3146 non-null   object 
 1   zone                  3146 non-null   object 
 2   gender                3146 non-null   object 
 3   deathdate             3146 non-null   object 
 4   age                   3146 non-null   float64
 5   avg_temperature       3146 non-null   float64
 6   max_temperature       3146 non-null   float64
 7   min_temperature       3146 non-null   float64
 8   precipitation         3146 non-null   float64
 9   latitude              3146 non-null   float64
 10  longitude             3146 non-null   float64
 11  location              3146 non-null   object 
 12  is_climate_sensitive  3146 non-null   int64  
dtypes: float64(7), int64(1), object(5)
memory usage: 319.6+ KB


In [131]:
max_loc_data = df['location'].apply(lambda x : len(x.split(',')))
max_loc_data.quantile(.75)

np.float64(5.0)

In [132]:
def preprocessor(data, fit=True, le=None, target=None, location_te=None, maxloc=int(max_loc_data.quantile(.75))):
    data = data.copy()

    data['zone'] = (data['zone'] == 'Peri_urban').astype(int)
    data['gender'] = (data['gender'] == 'Male').astype(int)

    data['age'] = data['age'].astype(int)
    data['age_group'] = pd.cut(
        data['age'],
        bins=[-1, 10, 20, 30, 40, 50, 60, 70, 80, 90, np.inf],
        labels=False
    ) + 1

    data["is_rain"] = (data["precipitation"] > 0).astype(int)

    data["is_heavy_rain"] = (
        data["precipitation"] > data["precipitation"].quantile(.75)
    ).astype(int)

    data["is_extreme_rain"] = (
        data["precipitation"] > data["precipitation"].quantile(.90)
    ).astype(int)

    data["precip_log"] = np.log1p(data["precipitation"])

    data["precip_per_temp"] = (
        data["precipitation"] /
        (abs(data["avg_temperature"]) + 1)
    )

    data["temp_rain_interaction"] = (
        data["avg_temperature"] *
        data["precipitation"]
    )

    data["avg_gt_min"] = (
        data["avg_temperature"] > data["min_temperature"]
    ).astype(int)

    data["avg_gt_max"] = (
        data["avg_temperature"] >= data["max_temperature"]
    ).astype(int)

    data["temp_range"] = (
        data["max_temperature"] -
        data["min_temperature"]
    )

    data["avg_temp_position"] = (
        (data["avg_temperature"] - data["min_temperature"]) /
        (data["max_temperature"] - data["min_temperature"] + 1e-6)
    )

    loc_cols = [f"location_{i+1}" for i in range(maxloc)]

    location_split = data["location"].str.split(",", expand=True)

    for i, col in enumerate(loc_cols):
        if i < location_split.shape[1]:
            data[col] = location_split[i].str.strip()
        else:
            data[col] = "Unknown"

    data['deathdate'] = pd.to_datetime(
        data['deathdate'],
        errors='coerce'
    )

    data['date'] = data['deathdate'].dt.day
    data['month'] = data['deathdate'].dt.month
    data['year'] = data['deathdate'].dt.year
    data['day_of_week'] = data['deathdate'].dt.dayofweek
    data['is_weekend'] = (data['day_of_week'] >= 5).astype(int)
    data['is_weekday'] = (data['day_of_week'] < 5).astype(int)
    data['quarter'] = data['deathdate'].dt.quarter
    data['day_of_year'] = data['deathdate'].dt.dayofyear
    data['week_of_year'] = (
        data['deathdate'].dt.isocalendar().week.astype(int)
    )

    if fit:
        if target is None:
            raise ValueError("target must be provided when fit=True")

        global_mean = data[target].mean()
        location_te = {}

        for col in loc_cols:
            stats = pd.DataFrame({
                'location': data[col],
                'target': data[target]
            }).groupby('location')['target'].agg(
                ['mean', 'count']
            )

            smoothing = 10

            stats['encoded'] = (
                (stats['count'] * stats['mean']) +
                (smoothing * global_mean)
            ) / (
                stats['count'] + smoothing
            )

            location_te[col] = {
                'mapping': stats['encoded'].to_dict(),
                'global_mean': global_mean
            }

            data[col] = data[col].map(
                location_te[col]['mapping']
            ).fillna(global_mean)

        data.drop(
            columns=['deathdate', 'location'],
            inplace=True
        )

        return data, location_te

    else:
        for col in loc_cols:
            mapping = location_te[col]['mapping']
            global_mean = location_te[col]['global_mean']

            data[col] = (
                data[col]
                .map(mapping)
                .fillna(global_mean)
            )

        data.drop(
            columns=['deathdate', 'location'],
            inplace=True
        )

        return data

In [133]:
df1, le_training = preprocessor(df, fit=True, target='is_climate_sensitive')

In [ ]:
X = df1.drop(columns=["is_climate_sensitive", "ID"])
y = df1["is_climate_sensitive"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [141]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ),
    
    "Extra Trees": ExtraTreesClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ),
    
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        random_state=42
    ),
    
    "XGBoost": XGBClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.75,
        colsample_bytree=0.75,
        random_state=42,
        eval_metric="logloss"
    )
}



results = []

for name, model in models.items():
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_prob)

    final_score = 0.6 * f1 + 0.4 * roc_auc

    results.append({
        "Model": name,
        "F1": f1,
        "ROC-AUC": roc_auc,
        "Final Score": final_score,
    })

results_df = pd.DataFrame(results).sort_values(
    "Final Score",
    ascending=False
).reset_index(drop=True)

print(results_df)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


                 Model        F1   ROC-AUC  Final Score
0    Gradient Boosting  0.821596  0.831430     0.825530
1        Random Forest  0.802339  0.811863     0.806149
2          Extra Trees  0.807468  0.797572     0.803510
3              XGBoost  0.791569  0.815211     0.801026
4  Logistic Regression  0.814398  0.751031     0.789051


In [142]:
tf = pd.read_csv('/kaggle/input/datasets/vedikagupta0/climate-risk-health-prediction-challenge/Test.csv')

In [143]:
tf1 = preprocessor(tf, fit=False, location_te=le_training, maxloc=5)

In [144]:
tf1.head()

,ID,zone,gender,age,avg_temperature,max_temperature,min_temperature,precipitation,latitude,longitude,...,location_5,date,month,year,day_of_week,is_weekend,is_weekday,quarter,day_of_year,week_of_year
0,ID_E760D84B,0,0,75,21.224387,24.632729,18.145633,4.295335,0.616667,33.500000,...,0.650668,24,11,2007,5,1,0,4,328,47
1,ID_6EDEA907,0,0,50,21.708596,25.184528,18.285274,1.376196,0.725422,33.098845,...,0.650668,23,1,2008,2,0,1,1,23,4
2,ID_B9FFC8D8,0,0,76,21.371149,24.584523,18.866490,2.336643,0.603194,33.542370,...,0.650668,14,3,2008,4,0,1,1,74,11
3,ID_74C6C94E,1,0,90,21.341990,25.188662,17.998317,4.703071,-0.590211,30.056939,...,0.650668,16,4,2008,2,0,1,2,107,16
4,ID_0E02825D,0,1,8,19.710391,22.905167,18.082628,4.066863,0.603194,33.542370,...,0.650668,21,4,2008,0,0,1,2,112,17


In [145]:
def make_submission(
    test_data,
    model,
    id_col="ID"
):
    ids = test_data.pop(id_col)

    pred_binary = model.predict(test_data)
    pred_prob = model.predict_proba(test_data)[:, 1]
    submission = pd.DataFrame({
        "ID": ids,
        "TargetF1": pred_binary.astype(int),
        "TargetRAUC": pred_prob
    })

    return submission

In [146]:
best_name = results_df.iloc[0]["Model"]
best_model = models[best_name]

X_full = df1.drop(columns=["is_climate_sensitive", "ID"])
y_full = df1["is_climate_sensitive"]

best_model.fit(X_full, y_full)

submission = make_submission(
    tf1.copy(),
    best_model
)

submission.to_csv("submission.csv", index=False)

print("Best model:", best_name)
print(submission.head())

Best model: Gradient Boosting
            ID  TargetF1  TargetRAUC
0  ID_E760D84B         1    0.750396
1  ID_6EDEA907         1    0.868798
2  ID_B9FFC8D8         1    0.590871
3  ID_74C6C94E         0    0.257114
4  ID_0E02825D         1    0.883512


In [147]:
submission.to_csv('climate-risk-fe.csv', index=False)